# Inferencia RAD-ALERT
Este notebook toma el dataset limpio (`data_cleaned.xlsx`), prepara el texto concatenando y normalizando las columnas relevantes,
ejecuta la inferencia con el modelo RAD-ALERT, y exporta el conjunto de datos de los reportes "Críticos" para entrenamiento.


In [6]:
# Ajuste liviano de versiones (evita reinstalar torch, que consume mucha RAM)
%pip install --upgrade --no-cache-dir "tokenizers==0.20.3" "transformers==4.46.3" Unidecode

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import re
from unidecode import unidecode
import warnings
warnings.filterwarnings('ignore')


In [8]:
# Cargar el dataset limpio
df = pd.read_excel('../data/data_cleaned.xlsx')
print(f"Total de registros originales: {len(df)}")

# Función para limpiar y normalizar texto al estilo RAD-ALERT
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower() # minúsculas
    text = unidecode(text) # sin acentos
    text = re.sub(r'[^a-z0-9\s]', ' ', text) # eliminar caracteres especiales
    text = re.sub(r'\s+', ' ', text).strip() # espacios extra
    return text

# Concatenar las columnas de interés
cols_texto = ['Datos Clínicos', 'Hallazgos', 'Opinión']
df['texto_informe'] = df[cols_texto].fillna('').agg(' '.join, axis=1)

# Aplicar normalización
df['texto_normalizado'] = df['texto_informe'].apply(normalize_text)

df[['texto_informe', 'texto_normalizado']].head()


Total de registros originales: 3977


,texto_informe,texto_normalizado
0,cefalea intensa con signos de alarma. descarta...,cefalea intensa con signos de alarma descartar...
1,tce. cefalea y emesis persistente. signos de ...,tce cefalea y emesis persistente signos de san...
2,cefalea con signos de alarma. dx mielomeningoc...,cefalea con signos de alarma dx mielomeningoce...
3,trauma facial y trauma cráneo encefálico. sig...,trauma facial y trauma craneo encefalico signo...
4,alteracion del estado de consiencia. surcos y...,alteracion del estado de consiencia surcos y e...


In [ ]:
# Configurar modelo RAD-ALERT
# IMPORTANTE: ajusta la ruta si mueves la carpeta del modelo.
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = Path(r"C:/Users/LENOVO/Documents/Universidad/Octavo semestre/PDG/RAD-ALERT/model")
print(f"Ruta modelo: {model_path}")
print(f"Existe ruta: {model_path.exists()}")

try:
    if not model_path.exists():
        raise FileNotFoundError(f"No existe la ruta del modelo: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), local_files_only=True)

    # Forzamos CPU para evitar errores de memoria en GPU
    device = torch.device('cpu')
    model.to(device)
    model.eval()
    print(f"Modelo cargado correctamente en {device}.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    print("Verifica la ruta y que la carpeta tenga config.json, tokenizer y model.safetensors.")


Ruta modelo: C:\Users\Isabella\Documents\Universidad\Octavo semestre\PDG\RAD-ALERT\model
Existe ruta: False


FileNotFoundError: No existe la ruta del modelo: C:\Users\Isabella\Documents\Universidad\Octavo semestre\PDG\RAD-ALERT\model
Ajusta la variable model_path a la ubicación correcta en tu máquina.

In [ ]:
# Función para predecir
def predict_critical(text):
    if not text.strip():
        return "No crítico", 0.0
        
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        # Ajusta el índice (1 o 0) según qué clase represente 'Crítico' en tu modelo
        prob_critico = probs[0][1].item() 
        
    pred = "Crítico" if prob_critico >= 0.5 else "No crítico"
    return pred, prob_critico

# Aplicar a todo el dataset
# Descomenta las siguientes líneas cuando el modelo esté configurado:
print("Iniciando inferencia...")
df['rad_prediccion'], df['rad_score'] = zip(*df['texto_normalizado'].apply(predict_critical))
print(df['rad_prediccion'].value_counts())


Iniciando inferencia...


NameError: name 'tokenizer' is not defined

In [ ]:
# Filtrar y Exportar
# Descomenta esto después de ejecutar la inferencia exitosamente.

df_criticos = df[df['rad_prediccion'] == 'Crítico'].copy()
print(f"Total de registros Críticos identificados por RAD-ALERT: {len(df_criticos)}")

# Guardar el dataset final como rad_criticos.xlsx
df_criticos.to_excel('../data/rad_criticos.xlsx', index=False)
print("Exportado exitosamente a '../data/rad_criticos.xlsx'")


Total de registros Críticos identificados por RAD-ALERT: 856
Exportado exitosamente a '../data/rad_criticos.xlsx'


In [ ]:
# Nubes de palabras por clase usando columnas: texto_normalizado y rad_prediccion
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Cambia esta ruta por tu archivo Excel
ruta_excel = '../data/rad_criticos.xlsx'

df_wc = pd.read_excel(ruta_excel)

# Validar columnas requeridas
columnas_requeridas = {'texto_normalizado', 'rad_prediccion'}
faltantes = columnas_requeridas - set(df_wc.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en el Excel: {faltantes}")

# Asegurar tipos
df_wc['texto_normalizado'] = df_wc['texto_normalizado'].fillna('').astype(str)
df_wc['rad_prediccion'] = df_wc['rad_prediccion'].fillna('').astype(str).str.strip()

# Separar textos por clase
texto_critico = ' '.join(df_wc.loc[df_wc['rad_prediccion'].str.lower() == 'crítico', 'texto_normalizado'])
texto_no_critico = ' '.join(df_wc.loc[df_wc['rad_prediccion'].str.lower() == 'no crítico', 'texto_normalizado'])

# Evitar error si una clase no tiene texto
if not texto_critico.strip():
    texto_critico = 'sin_datos'
if not texto_no_critico.strip():
    texto_no_critico = 'sin_datos'

# Crear nubes
wc_critico = WordCloud(width=1200, height=700, background_color='white', colormap='Reds').generate(texto_critico)
wc_no_critico = WordCloud(width=1200, height=700, background_color='white', colormap='Blues').generate(texto_no_critico)

# Mostrar ambas en una figura
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_critico, interpolation='bilinear')
axes[0].set_title('Nube de palabras - Crítico', fontsize=14)
axes[0].axis('off')

axes[1].imshow(wc_no_critico, interpolation='bilinear')
axes[1].set_title('Nube de palabras - No crítico', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

MemoryError: Unable to allocate 4.98 MiB for an array with shape (545, 1197) and data type uint64